In [1]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [3]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [4]:
t = load_queries("sql/02c_time_wait.sql")

EXPECTED = {"All": (20405666, 20205878, 199788),
            "Uber": (15356455, 15235488, 120967),
            "Lyft": (5049211, 4970390, 78821)}
def check(df, label):
    s = df.groupby("company")[["n_scoped", "n_valid", "n_excluded_negative_wait"]].sum()
    for comp, exp in EXPECTED.items():
        got = tuple(int(x) for x in s.loc[comp])
        print(label, comp, "OK" if got == exp else f"MISMATCH {got} vs {exp}")

hp = run(t, "hour_profile")
check(hp, "hour")
display(hp[hp.company == "All"])
display(hp.pivot(index="request_hour", columns="company", values=["p50_min", "p90_min", "pct_excluded"]))

hour All OK
hour Uber OK
hour Lyft OK


,company,request_hour,n_scoped,pct_of_company_volume,n_valid,n_excluded_negative_wait,pct_excluded,p50_min,p75_min,p90_min,p95_min
0,All,0,665891,3.263,664549,1342,0.202,4.167,6.233,9.300,11.767
1,All,1,465231,2.280,464422,809,0.174,4.233,6.400,9.550,12.000
2,All,2,343433,1.683,342314,1119,0.326,4.417,6.517,9.467,11.850
3,All,3,287760,1.410,283614,4146,1.441,4.600,6.583,9.350,11.567
4,All,4,296912,1.455,285226,11686,3.936,4.750,6.683,9.367,11.617
5,All,5,370523,1.816,356065,14458,3.902,4.417,6.167,8.633,10.783
6,All,6,602502,2.953,588488,14014,2.326,4.617,6.567,9.133,11.250
7,All,7,932409,4.569,911980,20429,2.191,4.983,7.333,10.267,12.600
8,All,8,1072184,5.254,1052496,19688,1.836,4.217,5.967,8.300,10.150
9,All,9,959350,4.701,941506,17844,1.860,3.483,4.817,6.633,8.217


p50_min               p90_min                 pct_excluded  \
company          All   Lyft   Uber     All    Lyft    Uber          All   
request_hour                                                              
0              4.167  4.250  4.133   9.300   8.567   9.567        0.202   
1              4.233  4.133  4.267   9.550   8.133  10.067        0.174   
2              4.417  4.267  4.483   9.467   8.267   9.917        0.326   
3              4.600  4.517  4.633   9.350   8.817   9.550        1.441   
4              4.750  4.767  4.750   9.367   9.067   9.517        3.936   
5              4.417  4.567  4.367   8.633   8.633   8.633        3.902   
6              4.617  4.867  4.533   9.133   9.283   9.067        2.326   
7              4.983  5.283  4.883  10.267  10.717  10.083        2.191   
8              4.217  4.533  4.117   8.300   8.850   8.083        1.836   
9              3.483  3.667  3.433   6.633   6.767   6.600        1.860   
10             3.350  3.517  3.300   6.533   6.583   6.517        1.680   
11             3.383  3.633  3.300   6.683   6.900   6.600        1.320   
12             3.517  3.817  3.433   7.050   7.317   6.950        1.352   
13             3.567  3.833  3.483   7.133   7.350   7.067        1.206   
14             3.783  4.067  3.683   7.750   8.050   7.633        0.919   
15             3.683  3.983  3.583   7.500   7.767   7.400        0.757   
16             3.583  3.833  3.517   7.200   7.333   7.150        0.629   
17             3.967  4.267  3.883   7.917   8.217   7.817        0.513   
18             3.933  4.183  3.867   7.733   7.900   7.683        0.363   
19             3.633  3.817  3.583   7.300   7.167   7.350        0.243   
20             3.533  3.650  3.500   7.417   7.000   7.567        0.185   
21             3.783  3.917  3.733   8.333   7.827   8.518        0.185   
22             4.067  4.167  4.033   8.750   8.233   8.950        0.169   
23             4.183  4.283  4.150   8.817   8.433   8.967        0.161   

                            
company        Lyft   Uber  
request_hour                
0             0.319  0.160  
1             0.247  0.147  
2             0.312  0.331  
3             1.447  1.438  
4             4.758  3.590  
5             5.509  3.263  
6             3.281  1.976  
7             3.809  1.613  
8             3.165  1.376  
9             2.895  1.519  
10            2.576  1.391  
11            1.952  1.112  
12            2.088  1.115  
13            1.847  1.001  
14            1.571  0.715  
15            1.281  0.596  
16            1.138  0.481  
17            0.930  0.387  
18            0.622  0.283  
19            0.363  0.205  
20            0.279  0.155  
21            0.310  0.144  
22            0.248  0.142  
23            0.210  0.144

In [5]:
wp = run(t, "weekday_profile")
check(wp, "weekday")
display(wp)

weekday All OK
weekday Uber OK
weekday Lyft OK


,company,request_isodow,dow_name,n_days,n_scoped,pct_of_company_volume,avg_trips_per_day,n_valid,n_excluded_negative_wait,pct_excluded,p50_min,p75_min,p90_min,p95_min
0,All,1,Monday,4,2276852,11.158,569213.0,2248464,28388,1.247,3.900,5.633,8.183,10.350
1,All,2,Tuesday,4,2487806,12.192,621952.0,2457677,30129,1.211,3.850,5.567,8.050,10.117
2,All,3,Wednesday,5,3266497,16.008,653299.0,3230518,35979,1.101,4.033,5.917,8.600,10.783
3,All,4,Thursday,5,3290379,16.125,658076.0,3251181,39198,1.191,3.817,5.517,7.950,10.017
4,All,5,Friday,5,3576142,17.525,715228.0,3539551,36591,1.023,3.817,5.467,7.767,9.717
5,All,6,Saturday,4,2978064,14.594,744516.0,2962998,15066,0.506,3.733,5.283,7.450,9.333
6,All,7,Sunday,4,2529926,12.398,632482.0,2515489,14437,0.571,3.967,5.717,8.317,10.533
7,Uber,1,Monday,4,1718266,11.189,429567.0,1701373,16893,0.983,3.817,5.533,8.100,10.350
8,Uber,2,Tuesday,4,1872103,12.191,468026.0,1854309,17794,0.950,3.767,5.467,7.983,10.117
9,Uber,3,Wednesday,5,2464432,16.048,492886.0,2442536,21896,0.888,3.983,5.917,8.717,11.000


In [6]:
hw = run(t, "hour_weekday_profile")
check(hw, "hour x weekday")
print("rows:", len(hw), "(expect 504)")
print(hw.groupby(["company", "sample_tier"]).size().unstack(fill_value=0))

cols = ["request_isodow", "request_hour", "n_valid", "pct_excluded", "avg_trips_per_day", "p50_min", "p90_min"]
pooled = hw[(hw.company == "All") & (hw.sample_tier == "3_adequate")]
print("Longest median waits (pooled adequate cells)"); display(pooled.sort_values("p50_min", ascending=False)[cols].head(10))
print("Shortest median waits (pooled adequate cells)"); display(pooled.sort_values("p50_min")[cols].head(10))

pv = (hw[hw.company.isin(["Uber", "Lyft"]) & (hw.sample_tier == "3_adequate")]
      .pivot_table(index=["request_isodow", "request_hour"], columns="company", values="p50_min").dropna())
gap = (pv["Lyft"] - pv["Uber"]) * 60
print(len(pv), "cells with both companies adequate | Lyft median above Uber in", int((gap > 0).sum()), "cells")
display(gap.describe().round(1).to_frame("Lyft minus Uber median gap (seconds)"))

hour x weekday All OK
hour x weekday Uber OK
hour x weekday Lyft OK
rows: 504 (expect 504)
sample_tier  3_adequate
company                
All                 168
Lyft                168
Uber                168
Longest median waits (pooled adequate cells)


,request_isodow,request_hour,n_valid,pct_excluded,avg_trips_per_day,p50_min,p90_min
49,3,1,90698,0.466,18225.0,6.067,12.900
50,3,2,75442,0.481,15161.0,5.908,12.267
31,2,7,143948,2.344,36851.0,5.500,11.333
79,4,7,180841,2.200,36982.0,5.417,10.833
51,3,3,63661,1.052,12868.0,5.333,10.650
4,1,4,29201,5.642,7737.0,5.250,11.233
103,5,7,175555,2.123,35873.0,5.233,10.467
3,1,3,21953,2.518,5630.0,5.133,10.783
55,3,7,159549,2.263,32649.0,5.100,10.300
52,3,4,57793,3.019,11918.0,5.083,10.117


Shortest median waits (pooled adequate cells)


,request_isodow,request_hour,n_valid,pct_excluded,avg_trips_per_day,p50_min,p90_min
83,4,11,134652,1.710,27399.0,3.233,6.483
82,4,10,142392,2.119,29095.0,3.250,6.400
106,5,10,142230,1.981,29021.0,3.267,6.400
35,2,11,103955,1.681,26433.0,3.300,6.667
130,6,10,115807,0.740,29168.0,3.317,6.250
84,4,12,135742,1.757,27634.0,3.333,6.733
105,5,9,162205,2.066,33125.0,3.333,6.267
131,6,11,125019,0.596,31442.0,3.333,6.383
34,2,10,111916,2.143,28592.0,3.333,6.583
117,5,21,197773,0.164,39619.0,3.367,6.833


168 cells with both companies adequate | Lyft median above Uber in 150 cells


,Lyft minus Uber median gap (seconds)
count,168.0
mean,13.3
std,19.1
min,-140.0
25%,9.0
50%,16.0
75%,22.0
max,54.0


In [10]:
es = run(t, "hour_exclusion_sensitivity")
display(es[es.company == "All"])
display(es.groupby("company").agg(max_pct_excluded=("pct_excluded", "max"),
                                  max_p50_width_sec=("p50_bound_width_sec", "max"),
                                  max_p90_width_sec=("p90_bound_width_sec", "max")))

,company,request_hour,n_scoped,n_valid,n_excluded_negative_wait,pct_excluded,p50_valid_min,p50_low_min,p50_high_min,p50_bound_width_sec,p50_level_low_pct,p50_level_high_pct,p90_valid_min,p90_low_min,p90_high_min,p90_bound_width_sec,p90_level_low_pct,p90_level_high_pct
0,All,0,665891,664549,1342,0.202,4.167,4.150,4.167,1.0,49.90,50.10,9.300,9.283,9.350,4.0,89.98,90.18
1,All,1,465231,464422,809,0.174,4.233,4.233,4.233,0.0,49.91,50.09,9.550,9.550,9.600,3.0,89.98,90.16
2,All,2,343433,342314,1119,0.326,4.417,4.400,4.417,1.0,49.84,50.16,9.467,9.450,9.567,7.0,89.97,90.29
3,All,3,287760,283614,4146,1.441,4.600,4.567,4.650,5.0,49.27,50.73,9.350,9.300,9.783,29.0,89.85,91.32
4,All,4,296912,285226,11686,3.936,4.750,4.633,4.883,15.0,47.95,52.05,9.367,9.250,10.850,96.0,89.59,93.69
5,All,5,370523,356065,14458,3.902,4.417,4.317,4.533,13.0,47.97,52.03,8.633,8.517,10.017,90.0,89.59,93.65
6,All,6,602502,588488,14014,2.326,4.617,4.550,4.700,9.0,48.81,51.19,9.133,9.067,9.850,47.0,89.76,92.14
7,All,7,932409,911980,20429,2.191,4.983,4.900,5.067,10.0,48.88,51.12,10.267,10.200,11.000,48.0,89.78,92.02
8,All,8,1072184,1052496,19688,1.836,4.217,4.167,4.267,6.0,49.06,50.94,8.300,8.250,8.783,32.0,89.81,91.68
9,All,9,959350,941506,17844,1.860,3.483,3.450,3.517,4.0,49.05,50.95,6.633,6.600,7.050,27.0,89.81,91.71


,max_pct_excluded,max_p50_width_sec,max_p90_width_sec
company,,,
All,3.936,15.0,96.0
Lyft,5.509,19.0,128.0
Uber,3.590,14.0,91.0


In [8]:
vc = run(t, "hourly_volume_curve")
curve = vc[vc.company == "All"].pivot(index="request_hour", columns="day_type", values="pct_of_company_daytype_volume")
display(curve)
for dt in curve.columns:
    print(dt, "| trough hour:", int(curve[dt].idxmin()), "| peak hour:", int(curve[dt].idxmax()))
display(vc[vc.request_hour.isin([0, 1, 2, 3])]
        .groupby(["company", "day_type"])["pct_of_company_daytype_volume"].sum().unstack().round(2))

day_type,Mon-Fri,Sat-Sun
request_hour,,
0,2.637,4.957
1,1.708,3.827
2,1.253,2.846
3,1.111,2.218
4,1.348,1.744
5,1.920,1.534
6,3.329,1.935
7,5.365,2.417
8,6.113,2.931


Mon-Fri | trough hour: 3 | peak hour: 18
Sat-Sun | trough hour: 5 | peak hour: 18


day_type,Mon-Fri,Sat-Sun
company,,
All,6.71,13.85
Lyft,6.83,16.03
Uber,6.67,13.14


In [9]:
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
hp.to_csv(out / "2c_hour_profile.csv", index=False)
wp.to_csv(out / "2c_weekday_profile.csv", index=False)
hw.to_csv(out / "2c_hour_weekday_profile.csv", index=False)
es.to_csv(out / "2c_hour_exclusion_sensitivity.csv", index=False)
vc.to_csv(out / "2c_hourly_volume_curve.csv", index=False)

In [11]:
chk = es.copy()
order_ok = bool(((chk.p50_low_min <= chk.p50_valid_min) & (chk.p50_valid_min <= chk.p50_high_min) &
                 (chk.p90_low_min <= chk.p90_valid_min) & (chk.p90_valid_min <= chk.p90_high_min)).all())
levels_ok = bool(((chk.p50_level_low_pct <= 50) & (chk.p50_level_high_pct >= 50) &
                  (chk.p90_level_low_pct <= 90) & (chk.p90_level_high_pct >= 90)).all())
print("rows:", len(chk), "(expect 72) | bound ordering OK:", order_ok, "| level ordering OK:", levels_ok)

def valid_q(hour, level_pct, comp):
    cond = "" if comp == "All" else f"AND company = '{comp}'"
    return con.execute(f"SELECT quantile_cont(wait_minutes, {level_pct / 100}) FROM trips_features "
                       f"WHERE is_wait_valid AND request_hour = {hour} {cond}").fetchone()[0]

for _, r in chk.sort_values("pct_excluded", ascending=False).head(3).iterrows():
    for label, lvl, val in [("p50 low", "p50_level_low_pct", "p50_low_min"), ("p50 high", "p50_level_high_pct", "p50_high_min"),
                            ("p90 low", "p90_level_low_pct", "p90_low_min"), ("p90 high", "p90_level_high_pct", "p90_high_min")]:
        v = valid_q(int(r["request_hour"]), r[lvl], r["company"])
        print(r["company"], int(r["request_hour"]), label, "| scenario:", r[val], "| level-based:", round(v, 3), "| diff:", round(abs(v - r[val]), 3))

rows: 72 (expect 72) | bound ordering OK: True | level ordering OK: True
Lyft 5 p50 low | scenario: 4.4 | level-based: 4.4 | diff: 0.0
Lyft 5 p50 high | scenario: 4.717 | level-based: 4.717 | diff: 0.0
Lyft 5 p90 low | scenario: 8.483 | level-based: 8.483 | diff: 0.0
Lyft 5 p90 high | scenario: 10.617 | level-based: 10.617 | diff: 0.0
Lyft 4 p50 low | scenario: 4.633 | level-based: 4.633 | diff: 0.0
Lyft 4 p50 high | scenario: 4.917 | level-based: 4.917 | diff: 0.0
Lyft 4 p90 low | scenario: 8.933 | level-based: 8.933 | diff: 0.0
Lyft 4 p90 high | scenario: 10.7 | level-based: 10.7 | diff: 0.0
All 4 p50 low | scenario: 4.633 | level-based: 4.633 | diff: 0.0
All 4 p50 high | scenario: 4.883 | level-based: 4.883 | diff: 0.0
All 4 p90 low | scenario: 9.25 | level-based: 9.25 | diff: 0.0
All 4 p90 high | scenario: 10.85 | level-based: 10.85 | diff: 0.0
